https://github.com/felixdrinkall/financial-news-dataset

## Objective

The objective of this notebook is to provide the tools needed to download the required data and assemble the datasets used in this GitHub repository.

## Dataset for BiLSTM/Bert/RAG

In [ ]:
import os
import json
import requests
import pandas as pd
import lzma

# CONFIG
repo_base_url = "https://raw.githubusercontent.com/FelixDrinkall/financial-news-dataset/main/data"
script_dir = os.path.dirname(os.path.abspath(__file__))
save_path = os.path.join(script_dir, "Data")
years = [2017, 2018, 2019, 2020, 2021, 2022, 2023]

os.makedirs(save_path, exist_ok=True)

# DOWNLOAD FILES FROM GITHUB
for year in years:
    filename = f"{year}_processed.json.xz"
    url = f"{repo_base_url}/{filename}"
    local_file = os.path.join(save_path, filename)

    if os.path.exists(local_file):
        print(f"Already exists: {filename}")
        continue

    print(f"Downloading: {filename}")
    r = requests.get(url, stream=True)
    r.raise_for_status()

    with open(local_file, "wb") as f:
        for chunk in r.iter_content(chunk_size=8192):
            if chunk:
                f.write(chunk)

print("\nDownload completed.\n")


# ROBUST JSON LOADING FUNCTION
def load_json_robust(path):
    try:
        return pd.read_json(path, lines=True, compression="infer")
    except ValueError:
        pass

    if path.endswith(".xz"):
        with lzma.open(path, "rt", encoding="utf-8") as f:
            text = f.read().strip()
    else:
        with open(path, "r", encoding="utf-8-sig") as f:
            text = f.read().strip()

    if not text:
        raise ValueError(f"Empty file: {path}")

    obj = json.loads(text)

    if isinstance(obj, list):
        return pd.DataFrame(obj)

    return pd.json_normalize(obj)

# READ + CONCATENATE
dfs = []
bad = []

for file in sorted(os.listdir(save_path)):
    if file.endswith(".json") or file.endswith(".json.xz"):
        fp = os.path.join(save_path, file)
        print("Reading:", file)

        try:
            df = load_json_robust(fp)
            year = int(file.split("_")[0])
            df["year"] = year
            dfs.append(df)
        except Exception as e:
            bad.append((file, str(e)))
            print("Error on", file, "->", e)

if dfs:
    final_df = pd.concat(dfs, ignore_index=True)
    print("\nFinal shape:", final_df.shape)

    output_file = os.path.join(save_path, "final_dataset.csv")
    final_df.to_csv(output_file, index=False)
    print("Final dataset saved to:", output_file)
else:
    final_df = pd.DataFrame()
    print("\nNo dataframe was loaded.")

if bad:
    print("\nProblematic files:")
    for b in bad:
        print(" -", b[0], ":", b[1])

print(final_df.head())

This script automatically downloads yearly financial news dataset files from a public GitHub repository, loads them into pandas DataFrames, and merges them into a single dataset.

### Workflow

1. Create a local folder named `Data` in the same directory as the script.
2. Download the compressed dataset files (`.json.xz`) for each year from GitHub if they are not already present locally.
3. Load each dataset using a robust JSON loader that supports both JSON Lines and standard JSON formats.
4. Extract the year from the filename and add it as a column in the DataFrame.
5. Concatenate all yearly DataFrames into a single dataset.
6. Save the final combined dataset as a CSV file inside the `Data` folder.
7. Print the final dataset shape and display the first rows for inspection.

If any file cannot be loaded correctly, it is recorded and reported at the end of the execution.

In [ ]:
# Official base column list
base_columns = [
    "date_publish",
    "description",
    "maintext",
    "title",
    "mentioned_companies",
    "related_companies",
    "industries",
    "sentiment",
    "emotion",
]

# Check for missing columns
missing = [c for c in base_columns if c not in final_df.columns]
print("Missing columns:", missing)

# Select only the base columns
base_df = final_df[base_columns].copy()

print("New dataset shape:", base_df.shape)
base_df.head()

The resulting dataset contains only the columns that are relevant for our project. 
These include the textual content of the articles and the key annotations used in the analysis, such as mentioned companies, industries, sentiment, and emotion labels.

In [ ]:
# Check missing values in the dataset
base_df.isna().sum()

# Remove rows where the description is missing
base_df = base_df.dropna(subset=["description"])

# Check missing values again after cleaning
print(base_df.isna().sum())

Before cleaning, the dataset contained 24 missing values in the `description` column, while all the other columns were complete.  
After removing the rows with missing descriptions, the dataset no longer contains missing values in any column.

In [ ]:
# Save the cleaned base dataset
base_df.to_json(
    "financial_news_base.jsonl",
    orient="records",
    lines=True
)

print("Dataset saved to financial_news_base.jsonl")

The cleaned base dataset is saved in JSONL format in the current directory so it can be easily used by the other components of the repository.

## Datasets for Dashboard

In [2]:
import os, json
import pandas as pd

def load_json_robust(path):
    # prova come JSON Lines
    try:
        return pd.read_json(path, lines=True)
    except ValueError:
        pass

    # fallback: JSON normale (lista o dict)
    with open(path, "r", encoding="utf-8-sig") as f:
        text = f.read().strip()

    # se ci sono righe vuote / spazi strani
    if not text:
        raise ValueError(f"File vuoto: {path}")

    obj = json.loads(text)

    # se è una lista di record
    if isinstance(obj, list):
        return pd.DataFrame(obj)

    # se è un dict con chiavi tipo columns/records ecc.
    return pd.json_normalize(obj)

path = "/content/drive/MyDrive/Ds"

dfs = []
bad = []

for file in sorted(os.listdir(path)):
    if file.endswith(".json"):
        fp = os.path.join(path, file)
        print("Leggo:", file)

        try:
            df = load_json_robust(fp)
            year = int(file.split("_")[0])
            df["year"] = year
            dfs.append(df)
        except Exception as e:
            bad.append((file, str(e)))
            print("  ❌ Errore su", file, "->", e)

final_df = pd.concat(dfs, ignore_index=True)
print("✅ Shape finale:", final_df.shape)

if bad:
    print("\nFile problematici:")
    for b in bad:
        print(" -", b[0], ":", b[1])

final_df.head()

Leggo: 2018_processed.json
Leggo: 2019_processed.json
Leggo: 2020_processed.json
Leggo: 2021_processed.json
Leggo: 2022_processed.json
Leggo: 2023_processed.json
✅ Shape finale: (70571, 170)


,authors,date_download,date_modify,date_publish,description,filename,image_url,language,localpath,maintext,...,curr_day_price_CSCO,prev_day_price_NIO,next_day_price_NIO,curr_day_price_NIO,prev_day_price_MRNA,next_day_price_MRNA,curr_day_price_MRNA,prev_day_price_UNH,next_day_price_UNH,curr_day_price_UNH
0,[],2018-06-24 14:03:26+00:00,None,2018-06-22 18:39:00,Morgan Stanley is telling its clients to pay a...,https%3A%2F%2Ffinance.yahoo.com%2Fnews%2Fmorga...,https://s.yimg.com/os/mit/media/p/common/image...,en,None,Morgan Stanley is telling its clients to pay a...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,[],2018-11-15 21:39:58+00:00,None,2018-11-15 00:00:00,"The FCC voted to grant ""market access"" request...",https%3A%2F%2Ffinance.yahoo.com%2Fnews%2Ffcc-o...,https://s.yimg.com/uu/api/res/1.2/qtgJvLeZydCO...,en,None,By David Shepardson\nWASHINGTON (Reuters - The...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,[],2018-12-31 02:00:12+00:00,None,2018-12-29 19:57:55,(Bloomberg) -- A lawsuit filed against Google ...,https%3A%2F%2Ffinance.yahoo.com%2Fnews%2Fgoogl...,https://s.yimg.com/uu/api/res/1.2/7P7R2HFr_Cu0...,en,None,(Bloomberg) -- A lawsuit filed against Google ...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,[],2018-09-13 00:56:51+00:00,None,2018-09-12 18:42:40,"With camera upgrades, better screens and more",https%3A%2F%2Ffinance.yahoo.com%2Fnews%2Fapple...,https://s.yimg.com/uu/api/res/1.2/Jb5AstRxGBd3...,en,None,Apple on Wednesday unveiled a trio of new iPho...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,[],2018-04-25 02:02:32+00:00,None,2018-04-23 20:03:16,Google is reporting on Monday.,https%3A%2F%2Ffinance.yahoo.com%2Fnews%2Falpha...,https://s.yimg.com/uu/api/res/1.2/7rqX9riG7WX0...,en,None,"Google’s (GOOG, GOOGL) parent company Alphabet...",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [3]:
# Lista colonne base ufficiali
base_columns = [
    "date_publish",
    "description",
    "maintext",
    "title",
    "mentioned_companies",
    "related_companies",
    "industries",
    "sentiment",
    "emotion",
]

# Controllo colonne mancanti
missing = [c for c in base_columns if c not in final_df.columns]
print("Colonne mancanti:", missing)

# Selezione solo colonne base
base_df = final_df[base_columns].copy()

print("Nuova shape:", base_df.shape)
base_df.head()

Colonne mancanti: []
Nuova shape: (70571, 9)


,date_publish,description,maintext,title,mentioned_companies,related_companies,industries,sentiment,emotion
0,2018-06-22 18:39:00,Morgan Stanley is telling its clients to pay a...,Morgan Stanley is telling its clients to pay a...,Morgan Stanley sees 'a pattern forming' of the...,[GOOGL],"[IGLD, RAMP, NSR, TWTR, ACXM, COR, PINS, META,...",[7375],"{'negative': 0.0005622319295071065, 'neutral':...","{'neutral': 0.9212942719459534, 'surprise': 0...."
1,2018-11-15 00:00:00,"The FCC voted to grant ""market access"" request...",By David Shepardson\nWASHINGTON (Reuters - The...,"SpaceX, TeleSat Canada bids get U.S. nod to ex...","[TSLA, MU]","[IKGH, CLLS, EBIO, GMAB, GBS, WIMI, WALD, FGEN...","[3674, 9999]","{'negative': 0.00028582950471900403, 'neutral'...","{'neutral': 0.8668617606163025, 'joy': 0.05769..."
2,2018-12-29 19:57:55,(Bloomberg) -- A lawsuit filed against Google ...,(Bloomberg) -- A lawsuit filed against Google ...,Google Wins Dismissal of Suit Over Facial Reco...,[GOOGL],"[IGLD, RAMP, NSR, TWTR, ACXM, COR, PINS, META,...",[7375],"{'negative': 0.23072107136249542, 'neutral': 0...","{'anger': 0.7141016125679016, 'neutral': 0.202..."
3,2018-09-12 18:42:40,"With camera upgrades, better screens and more",Apple on Wednesday unveiled a trio of new iPho...,"Apple Unveils iPhone XS, iPhone XS Max and Che...",[AAPL],"[nan, TDC, HPQ, SCKT, OMCL, CRAY, ZEPP, IBM, S...",[3571],"{'negative': 0.005342656280845404, 'neutral': ...","{'neutral': 0.9592552185058594, 'surprise': 0...."
4,2018-04-23 20:03:16,Google is reporting on Monday.,"Google’s (GOOG, GOOGL) parent company Alphabet...",Alphabet Q1 2018 earnings,"[GOOGL, INTC, META, AAPL]","[IGLD, TDC, NSAT, CRAY, TCX, FLEX, SMCI, WBMD,...","[3571, 3679, 7375]","{'negative': 0.00031546535319648683, 'neutral'...","{'neutral': 0.6759750247001648, 'surprise': 0...."


In [4]:
base_df.isna().sum()

,0
date_publish,0
description,24
maintext,0
title,0
mentioned_companies,0
related_companies,0
industries,0
sentiment,0
emotion,0


In [5]:
base_df = base_df.dropna(subset=["description"])
print(base_df.isna().sum())

date_publish           0
description            0
maintext               0
title                  0
mentioned_companies    0
related_companies      0
industries             0
sentiment              0
emotion                0
dtype: int64


In [6]:
base_df.to_json(
    "/content/drive/MyDrive/Ds/financial_news_base.jsonl",
    orient="records",
    lines=True
)

In [7]:
for col in base_df.columns:
    print(f"{col}:")
    print(base_df[col].apply(type).value_counts())
    print("------")

date_publish:
date_publish
<class 'str'>    70547
Name: count, dtype: int64
------
description:
description
<class 'str'>    70547
Name: count, dtype: int64
------
maintext:
maintext
<class 'str'>    70547
Name: count, dtype: int64
------
title:
title
<class 'str'>    70547
Name: count, dtype: int64
------
mentioned_companies:
mentioned_companies
<class 'list'>    70547
Name: count, dtype: int64
------
related_companies:
related_companies
<class 'list'>    70547
Name: count, dtype: int64
------
industries:
industries
<class 'list'>    70547
Name: count, dtype: int64
------
sentiment:
sentiment
<class 'dict'>    70547
Name: count, dtype: int64
------
emotion:
emotion
<class 'dict'>    70547
Name: count, dtype: int64
------


In [9]:
import ast

base_df["mentioned_companies"] = base_df["mentioned_companies"].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else x
)

all_companies = base_df["mentioned_companies"].explode().unique()
print(all_companies)

['GOOGL' 'TSLA' 'MU' 'AAPL' 'INTC' 'META' 'T' 'V' 'AMZN' 'WMT' 'NFLX'
 'JNJ' 'C' 'BA' 'MSFT' 'DIS' 'VZ' 'MA' 'GS' 'GE' 'KO' 'QCOM' 'NVDA' 'ADBE'
 'BAC' 'WFC' 'BABA' 'JPM' 'CRM' 'BRK' 'SQ' 'PYPL' 'PFE' 'XOM' 'CVX' 'ROKU'
 'ORCL' 'SHOP' 'COST' 'CMCSA' 'AVGO' 'HD' 'PG' 'LLY' 'MRK' 'CSCO' 'NIO'
 'MRNA' 'UNH']


In [10]:
i = 45
print(base_df["mentioned_companies"][i])
print(base_df["maintext"][i])

['TSLA']
SpaceX is gearing up in its effort to build a rocket that could carry mankind to Mars and beyond. This week, the company posted its first job posting dedicated to the project.
As part of its grand vision, SpaceX is designing the Big Falcon Rocket, or BFR, as a launch vehicle capable of carrying people to Mars. CEO Elon Musk envisions that the technology will eventually replace the Falcon 9 and even shuttle passengers between New York and Los Angeles in about 25 minutes.
To achieve this, SpaceX is hiring a “build engineer” with experience in aerospace and mechanical engineering who can “work long hours and weekends” whenever needed. “Working directly in the Vehicle Engineering group, the goal of this team is to investigate, test, and develop new hardware, software, and automation efforts capable of supporting advanced metallic and composite joining methods for the BFR,” said the job description, which listed no salary range or application deadline.
Announced in early 2017, the 

In [11]:
base_df["sentiment_label"] = base_df["sentiment"].apply(
    lambda x: max(x.items(), key=lambda item: item[1])[0]
)

print(base_df["sentiment_label"].value_counts())

sentiment_label
positive    39098
neutral     19028
negative    12421
Name: count, dtype: int64


In [12]:
all_industries = set(i for sublist in base_df["industries"] for i in sublist)
print(all_industries)

{6021, 6029.0, 9999, 2834, 2844, 7841, 2086, 6324, '3841', 3511, 5311, 6719, 6211, '6799', 7370, 4812, 4813, 7372, 7375, 3663, 5330, 7379, 3670, 3674, 5211, 7389, '6331', 3679, '2834', 4833, 2911, '9999', '3721', 4841, 3571}


In [13]:
sic_2digit_dict = {
    1: "Agriculture, Forestry, Fishing",
    10: "Mining",
    15: "Construction",
    20: "Food and Kindred Products",
    27: "Printing and Publishing",
    28: "Chemicals and Allied Products",
    29: "Petroleum and Coal Products",
    35: "Industrial Machinery and Equipment",
    36: "Electronic and Electrical Equipment",
    37: "Transportation Equipment",
    38: "Instruments and Related Products",
    48: "Communications",
    52: "Building Materials & Hardware",
    53: "General Merchandise Stores",
    60: "Depository Institutions (Banks)",
    62: "Security & Commodity Brokers",
    63: "Insurance Carriers",
    67: "Holding and Investment Offices",
    73: "Business Services",
    78: "Motion Pictures",
    99: "Nonclassifiable Establishments"
}

base_df["industries"] = base_df["industries"].apply(
    lambda lst: [int(float(x)) for x in lst]
)

base_df["industry_2digit"] = base_df["industries"].apply(
    lambda lst: [int(str(code)[:2]) for code in lst]
)

base_df["industry_macro"] = base_df["industry_2digit"].apply(
    lambda lst: [sic_2digit_dict.get(code, "Unknown") for code in lst]
)


In [14]:
for i in range(20):
    print(base_df["sentiment"][i])

{'negative': 0.0005622319295071065, 'neutral': 0.006970268674194813, 'positive': 0.99246746301651}
{'negative': 0.00028582950471900403, 'neutral': 0.0016880716430023313, 'positive': 0.9980260729789734}
{'negative': 0.23072107136249542, 'neutral': 0.36165717244148254, 'positive': 0.407621830701828}
{'negative': 0.005342656280845404, 'neutral': 0.16495321691036224, 'positive': 0.8297041654586792}
{'negative': 0.00031546535319648683, 'neutral': 5.8617933973437175e-05, 'positive': 0.9996259212493896}
{'negative': 0.00020293964189477265, 'neutral': 0.9843824505805969, 'positive': 0.015414595603942871}
{'negative': 0.0007130949525162578, 'neutral': 0.99896240234375, 'positive': 0.0003245453117415309}
{'negative': 0.005509575828909874, 'neutral': 0.9938179850578308, 'positive': 0.0006724508129991591}
{'negative': 0.0056778560392558575, 'neutral': 0.9367163777351379, 'positive': 0.05760573223233223}
{'negative': 0.0018842422869056463, 'neutral': 0.0003500903840176761, 'positive': 0.99776566028